In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Educational Disclaimer
print("=" * 70)
print("PROJECT SCOPE: Strictly educational dataset exploration.")
print("This model is not a medical diagnostic or screening tool.")
print("=" * 70)

In [ ]:
df = pd.read_csv("diabetes.csv")

# Biological features where 0 represents missing records
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("Zero counts prior to cleaning:")
print((df[zero_as_missing] == 0).sum())

# Replace biological 0s with NaN
df_cleaned = df.copy()
df_cleaned[zero_as_missing] = df_cleaned[zero_as_missing].replace(0, np.nan)

print("\nMissing values (NaN) after replacement:")
print(df_cleaned.isnull().sum())

In [ ]:
# 1. Target Class Balance
plt.figure(figsize=(6, 4))
sns.countplot(data=df_cleaned, x="Outcome", palette="Set2")
plt.title("Target Distribution (0 = Negative, 1 = Positive)")
plt.xlabel("Recorded Outcome")
plt.ylabel("Record Count")
plt.show()

# Class ratio calculation
pos_pct = (df_cleaned["Outcome"].sum() / len(df_cleaned)) * 100
print(
    f"Class distribution: {100 - pos_pct:.1f}% Negative, {pos_pct:.1f}% Positive"
)

# 2. Glucose & BMI Distributions by Outcome
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(
    data=df_cleaned,
    x="Glucose",
    hue="Outcome",
    kde=True,
    ax=axes[0],
    palette="Set2",
)
axes[0].set_title("Glucose Distribution by Outcome")

sns.boxplot(
    data=df_cleaned, x="Outcome", y="BMI", ax=axes[1], palette="Set2"
)
axes[1].set_title("BMI Distribution by Outcome")
plt.tight_layout()
plt.show()

# 3. Correlation Heatmap
plt.figure(figsize=(9, 6))
sns.heatmap(df_cleaned.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Pearson Correlation Matrix (Non-Causal Associations)")
plt.show()

In [ ]:
X = df_cleaned.drop(columns=["Outcome"])
y = df_cleaned["Outcome"]

# Stratified split to preserve class proportions in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

In [ ]:
# 1. Baseline Model (Dummy Classifier)
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

# 2. Logistic Regression Pipeline (Median Imputation + Standard Scaling)
pipe_lr = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(max_iter=1000, class_weight="balanced"),
        ),
    ]
)

pipe_lr.fit(X_train, y_train)
lr_pred = pipe_lr.predict(X_test)
lr_probs = pipe_lr.predict_proba(X_test)[:, 1]

In [ ]:
cm = confusion_matrix(y_test, lr_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn)  # True Positive Rate / Recall
specificity = tn / (tn + fp)  # True Negative Rate
precision = precision_score(y_test, lr_pred)

print(f"--- Model Performance vs Dummy Baseline ---")
print(f"Baseline Accuracy:    {dummy.score(X_test, y_test):.3f}")
print(f"Logistic Sensitivity: {sensitivity:.3f} (Detection of positive cases)")
print(f"Logistic Specificity: {specificity:.3f} (Correctly identified negative cases)")
print(f"Logistic Precision:   {precision:.3f}")
print("\nClassification Report:\n", classification_report(y_test, lr_pred))

# Confusion Matrix Heatmap
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred Neg", "Pred Pos"],
    yticklabels=["Actual Neg", "Actual Pos"],
)
plt.title("Logistic Regression Confusion Matrix")
plt.ylabel("Ground Truth")
plt.xlabel("Prediction")
plt.show()

print(
    f"Error Breakdown:\n- False Negatives (Missed positives): {fn}\n- False Positives (Unwarranted flags): {fp}"
)

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, lr_probs, n_bins=5)

plt.figure(figsize=(6, 5))
plt.plot(
    prob_pred, prob_true, marker="o", label="Logistic Regression Pipeline"
)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect Calibration")
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Fraction of Positives")
plt.title("Calibration Curve (Reliability Diagram)")
plt.legend()
plt.show()

In [ ]:
coeffs = pipe_lr.named_steps["classifier"].coef_[0]
feature_names = X.columns

importance_df = pd.DataFrame(
    {"Feature": feature_names, "Coefficient": coeffs}
).sort_values(by="Coefficient", ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=importance_df, x="Coefficient", y="Feature", palette="viridis")
plt.title("Logistic Regression Standardized Coefficients")
plt.xlabel("Log-Odds Impact (Association Only, Not Medical Causation)")
plt.show()

In [ ]:
# Synthetic sample representing hypothetical measurements
synthetic_patient = pd.DataFrame(
    [
        {
            "Pregnancies": 2,
            "Glucose": 138.0,
            "BloodPressure": 72.0,
            "SkinThickness": 35.0,
            "Insulin": 120.0,
            "BMI": 32.4,
            "DiabetesPedigreeFunction": 0.55,
            "Age": 33,
        }
    ]
)

predicted_risk = pipe_lr.predict_proba(synthetic_patient)[0, 1]
print("--- Synthetic Example Inference ---")
print(f"Calculated Statistical Risk Score: {predicted_risk * 100:.2f}%")
print("Notice: This number is a mathematical output from an unvalidated classroom model,")
print("not a clinical diagnosis.")